# Flanker任务：图像生成与VGG一致性分类

本notebook演示如何：
1. 生成Flanker任务的视觉刺激图像
2. 使用VGG模型进行训练
3. 判断刺激的一致性（congruent vs incongruent）

## 1. 导入必要的库

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, classification_report
import os
import random

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# 设备配置
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

## 2. Flanker任务图像生成器

In [ ]:
class FlankerStimulusGenerator:
    """
    Flanker任务刺激生成器
    
    生成包含目标鸟和干扰鸟的图像，用于注意力研究
    """
    
    def __init__(self, img_size=128, bird_size=20):
        self.img_size = img_size
        self.bird_size = bird_size
        self.directions = ['L', 'R', 'U', 'D']  # 左右上下
        self.direction_arrows = {
            'L': '◀',
            'R': '▶', 
            'U': '▲',
            'D': '▼'
        }
        
        # 布局定义：干扰项相对于目标的位置
        self.layouts = {
            'horizontal': [(-2, 0), (-1, 0), (1, 0), (2, 0)],
            'vertical': [(0, -2), (0, -1), (0, 1), (0, 2)],
            'cross': [(-1, 0), (1, 0), (0, -1), (0, 1)],
            'diagonal': [(-1, -1), (1, 1), (-1, 1), (1, -1)]
        }
        self.spacing = 25  # 鸟之间的间距
    
    def draw_bird(self, draw, center_x, center_y, direction, color='blue'):
        """
        绘制一个指向特定方向的鸟（用箭头表示）
        """
        arrow = self.direction_arrows[direction]
        
        # 使用更大的字体绘制箭头
        try:
            font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", self.bird_size)
        except:
            font = ImageFont.load_default()
        
        # 获取文本边界框
        bbox = draw.textbbox((0, 0), arrow, font=font)
        text_width = bbox[2] - bbox[0]
        text_height = bbox[3] - bbox[1]
        
        # 计算绘制位置（居中）
        x = center_x - text_width // 2
        y = center_y - text_height // 2
        
        # 绘制箭头
        draw.text((x, y), arrow, font=font, fill=color)
    
    def generate_stimulus(self, target_dir, flanker_dir, layout='horizontal', 
                      target_pos=None, bg_color='white'):
        """
        生成单个Flanker刺激图像
        
        参数:
            target_dir: 目标方向 ('L', 'R', 'U', 'D')
            flanker_dir: 干扰项方向 ('L', 'R', 'U', 'D')
            layout: 布局类型
            target_pos: 目标位置 (x, y)，默认为图像中心
            bg_color: 背景颜色
            
        返回:
            PIL Image对象
        """
        # 创建空白图像
        img = Image.new('RGB', (self.img_size, self.img_size), bg_color)
        draw = ImageDraw.Draw(img)
        
        # 设置目标位置（默认为中心）
        if target_pos is None:
            target_pos = (self.img_size // 2, self.img_size // 2)
        
        # 绘制目标（中心，用红色表示）
        self.draw_bird(draw, target_pos[0], target_pos[1], target_dir, color='red')
        
        # 计算干扰项位置
        offsets = self.layouts[layout]
        for offset in offsets:
            flanker_pos = (
                target_pos[0] + offset[0] * self.spacing,
                target_pos[1] + offset[1] * self.spacing
            )
            # 绘制干扰项（周围，用蓝色表示）
            self.draw_bird(draw, flanker_pos[0], flanker_pos[1], flanker_dir, color='blue')
        
        return img
    
    def generate_dataset(self, n_samples=1000, test_ratio=0.2):
        """
        生成完整的Flanker数据集
        
        返回:
            images: 图像数组 (n_samples, 3, img_size, img_size)
            labels: 一致性标签 (0=congruent, 1=incongruent)
            metadata: 元数据字典
        """
        images = []
        labels = []
        metadata = {
            'target_dirs': [],
            'flanker_dirs': [],
            'layouts': []
            'positions': []
        }
        
        for i in range(n_samples):
            # 随机选择参数
            target_dir = random.choice(self.directions)
            flanker_dir = random.choice(self.directions)
            layout = random.choice(list(self.layouts.keys()))
            
            # 判断一致性
            is_congruent = (target_dir == flanker_dir)
            label = 0 if is_congruent else 1
            
            # 生成图像
            img = self.generate_stimulus(target_dir, flanker_dir, layout)
            
            # 转换为numpy数组并归一化
            img_array = np.array(img).astype(np.float32) / 255.0
            # 转换为CHW格式 (Channels, Height, Width)
            img_array = np.transpose(img_array, (2, 0, 1))
            
            images.append(img_array)
            labels.append(label)
            
            # 保存元数据
            metadata['target_dirs'].append(target_dir)
            metadata['flanker_dirs'].append(flanker_dir)
            metadata['layouts'].append(layout)
            metadata['positions'].append((self.img_size//2, self.img_size//2))
        
        return np.array(images), np.array(labels), metadata

## 3. 生成示例图像并可视化

In [ ]:
# 创建生成器
generator = FlankerStimulusGenerator(img_size=128, bird_size=24)

# 生成示例图像
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Flanker任务刺激示例', fontsize=16)

# Congruent examples (目标=干扰)
congruent_examples = [
    ('L', 'L', 'horizontal'),
    ('R', 'R', 'vertical'),
    ('U', 'U', 'cross'),
    ('D', 'D', 'diagonal')
]

for i, (t_dir, f_dir, layout) in enumerate(congruent_examples):
    img = generator.generate_stimulus(t_dir, f_dir, layout)
    axes[0, i].imshow(img)
    axes[0, i].set_title(f'Congruent: 目标={t_dir}, 干扰={f_dir}\n布局={layout}', fontsize=10)
    axes[0, i].axis('off')

# Incongruent examples (目标≠干扰)
incongruent_examples = [
    ('L', 'R', 'horizontal'),
    ('R', 'L', 'vertical'),
    ('U', 'D', 'cross'),
    ('D', 'U', 'diagonal')
]

for i, (t_dir, f_dir, layout) in enumerate(incongruent_examples):
    img = generator.generate_stimulus(t_dir, f_dir, layout)
    axes[1, i].imshow(img)
    axes[1, i].set_title(f'Incongruent: 目标={t_dir}, 干扰={f_dir}\n布局={layout}', fontsize=10)
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

print("说明：")
print("- 红色箭头：目标鸟（需要关注的）")
print("- 蓝色箭头：干扰鸟（需要抑制的）")
print("- Congruent: 目标和干扰项方向相同")
print("- Incongruent: 目标和干扰项方向不同")

## 4. 生成训练数据集

In [ ]:
# 生成数据集
print("正在生成Flanker任务数据集...")
n_samples = 2000
images, labels, metadata = generator.generate_dataset(n_samples=n_samples)

print(f"数据集生成完成！")
print(f"总样本数: {len(images)}")
print(f"图像形状: {images.shape}")
print(f"标签分布:")
print(f"  - Congruent (0): {np.sum(labels == 0)} ({np.mean(labels == 0)*100:.1f}%)")
print(f"  - Incongruent (1): {np.sum(labels == 1)} ({np.mean(labels == 1)*100:.1f}%)")

## 5. 创建PyTorch数据集类

In [ ]:
class FlankerDataset(Dataset):
    """
    Flanker任务数据集
    """
    
    def __init__(self, images, labels, metadata=None, transform=None):
        self.images = images
        self.labels = labels
        self.metadata = metadata
        self.transform = transform
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

# 数据增强
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 创建数据集
dataset = FlankerDataset(images, labels, metadata, transform=None)

# 划分训练集和测试集
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

# 应用不同的transform
train_dataset.dataset.transform = train_transform
test_dataset.dataset.transform = test_transform

print(f"训练集大小: {len(train_dataset)}")
print(f"测试集大小: {len(test_dataset)}")

## 6. 创建VGG模型

In [ ]:
class VGGCongruenceClassifier(nn.Module):
    """
    基于VGG的一致性分类器
    
    使用预训练的VGG16作为特征提取器，
    添加自定义分类头判断Flanker刺激的一致性
    """
    
    def __init__(self, num_classes=2, pretrained=True):
        super(VGGCongruenceClassifier, self).__init__()
        
        # 加载预训练的VGG16
        vgg = models.vgg16(pretrained=pretrained)
        
        # 提取特征提取部分（去掉最后的分类层）
        self.features = vgg.features
        
        # 获取VGG的avgpool层
        self.avgpool = vgg.avgpool
        
        # 计算特征维度
        self.feature_dim = 512 * 4 * 4  # VGG16在128x128输入下的特征维度
        
        # 自定义分类头
        self.classifier = nn.Sequential(
            nn.Linear(self.feature_dim, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )
        
        # 冻结预训练层（可选）
        for param in self.features.parameters():
            param.requires_grad = False
        for param in self.avgpool.parameters():
            param.requires_grad = False
    
    def forward(self, x):
        # 特征提取
        x = self.features(x)
        x = self.avgpool(x)
        
        # 展平
        x = torch.flatten(x, 1)
        
        # 分类
        x = self.classifier(x)
        
        return x

# 创建模型
model = VGGCongruenceClassifier(num_classes=2, pretrained=True).to(device)

# 打印模型结构
print("模型结构:")
print(model)

# 统计可训练参数
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n总参数: {total_params:,}")
print(f"可训练参数: {trainable_params:,}")

## 7. 训练配置

In [ ]:
# 训练参数
batch_size = 32
learning_rate = 0.001
num_epochs = 20

# 创建数据加载器
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 学习率调度器
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

print("训练配置:")
print(f"  - 批次大小: {batch_size}")
print(f"  - 学习率: {learning_rate}")
print(f"  - 训练轮数: {num_epochs}")
print(f"  - 优化器: Adam")
print(f"  - 学习率调度: StepLR (每5轮减半)")

## 8. 训练函数

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """
    训练一个epoch
    """
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (images, labels) in enumerate(dataloader):
        images, labels = images.to(device), labels.to(device)
        
        # 前向传播
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # 反向传播和优化
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # 统计
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100 * correct / total
    
    return epoch_loss, epoch_acc

def evaluate(model, dataloader, criterion, device):
    """
    评估模型
    """
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100 * correct / total
    
    return epoch_loss, epoch_acc, all_predictions, all_labels

## 9. 训练模型

In [ ]:
# 训练循环
train_losses = []
train_accs = []
test_losses = []
test_accs = []

best_test_acc = 0.0

print("开始训练...")
print("="*50)

for epoch in range(num_epochs):
    # 训练
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # 评估
    test_loss, test_acc, _, _ = evaluate(model, test_loader, criterion, device)
    
    # 更新学习率
    scheduler.step()
    
    # 记录历史
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    test_losses.append(test_loss)
    test_accs.append(test_acc)
    
    # 保存最佳模型
    if test_acc > best_test_acc:
        best_test_acc = test_acc
        torch.save(model.state_dict(), 'best_vgg_congruence_model.pth')
    
    # 打印进度
    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"  训练 - Loss: {train_loss:.4f}, Acc: {train_acc:.2f}%")
    print(f"  测试 - Loss: {test_loss:.4f}, Acc: {test_acc:.2f}%")
    print(f"  学习率: {optimizer.param_groups[0]['lr']:.6f}")
    print("-"*30)

print("\n训练完成！")
print(f"最佳测试准确率: {best_test_acc:.2f}%")

## 10. 可视化训练过程

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 损失曲线
axes[0].plot(range(1, num_epochs+1), train_losses, 'b-', label='训练损失', linewidth=2)
axes[0].plot(range(1, num_epochs+1), test_losses, 'r-', label='测试损失', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('训练和测试损失', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# 准确率曲线
axes[1].plot(range(1, num_epochs+1), train_accs, 'b-', label='训练准确率', linewidth=2)
axes[1].plot(range(1, num_epochs+1), test_accs, 'r-', label='测试准确率', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('训练和测试准确率', fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. 模型评估与详细分析

In [ ]:
# 加载最佳模型
model.load_state_dict(torch.load('best_vgg_congruence_model.pth'))

# 在测试集上评估
test_loss, test_acc, predictions, true_labels = evaluate(
    model, test_loader, criterion, device
)

print("="*50)
print("最终模型评估")
print("="*50)
print(f"测试损失: {test_loss:.4f}")
print(f"测试准确率: {test_acc:.2f}%")
print()

# 详细分类报告
print("分类报告:")
print(classification_report(
    true_labels, 
    predictions, 
    target_names=['Congruent', 'Incongruent'],
    digits=4
))

## 12. 可视化预测结果

In [ ]:
# 获取一些测试样本进行可视化
model.eval()

# 收集一些预测结果
sample_images = []
sample_true_labels = []
sample_pred_labels = []
sample_confs = []

with torch.no_grad():
    for images, labels in test_loader:
        if len(sample_images) >= 16:  # 只收集16个样本
            break
            
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        probs = F.softmax(outputs, dim=1)
        confidences, predicted = torch.max(probs, 1)
        
        # 反归一化图像用于显示
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        denorm_images = images * std + mean
        denorm_images = torch.clamp(denorm_images, 0, 1)
        
        sample_images.extend(denorm_images.cpu().numpy())
        sample_true_labels.extend(labels.cpu().numpy())
        sample_pred_labels.extend(predicted.cpu().numpy())
        sample_confs.extend(confidences.cpu().numpy())

# 可视化
fig, axes = plt.subplots(4, 4, figsize=(16, 16))
fig.suptitle('模型预测结果示例', fontsize=16)

class_names = ['Congruent', 'Incongruent']

for i, ax in enumerate(axes.flat):
    # 显示图像（转回HWC格式）
    img = np.transpose(sample_images[i], (1, 2, 0))
    ax.imshow(img)
    
    # 获取真实和预测标签
    true_label = class_names[sample_true_labels[i]]
    pred_label = class_names[sample_pred_labels[i]]
    confidence = sample_confs[i] * 100
    
    # 判断是否正确
    is_correct = sample_true_labels[i] == sample_pred_labels[i]
    color = 'green' if is_correct else 'red'
    
    # 设置标题
    title = f'True: {true_label}\nPred: {pred_label} ({confidence:.1f}%)'
    ax.set_title(title, color=color, fontsize=10, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

print("图例说明：")
print("- 绿色标题：预测正确")
print("- 红色标题：预测错误")
print("- 百分比：模型预测的置信度")

## 13. 模型如何判断一致性？

In [ ]:
print("="*60)
print("模型判断一致性的机制")
print("="*60)
print()

print("1. 输入处理:")
print("   - 输入: 128×128×3 的Flanker刺激图像")
print("   - 包含: 1个目标鸟(红色) + 4个干扰鸟(蓝色)")
print()

print("2. 特征提取 (VGG16):")
print("   - 卷积层: 提取边缘、纹理、形状等视觉特征")
print("   - 池化层: 压缩空间信息，突出重要特征")
print("   - 输出: 512个4×4的特征图")
print()

print("3. 特征分析:")
print("   - Congruent样本: 目标和干扰项方向相同")
print("     → 所有箭头指向同一方向")
print("     → 视觉特征更加一致和规则")
print("   - Incongruent样本: 目标和干扰项方向不同")
print("     → 箭头指向不同方向")
print("     → 视觉特征存在冲突和不规则")
print()

print("4. 分类决策:")
print("   - 全连接层: 学习特征与一致性的映射关系")
print("   - 输出: 2个类别的logits")
print("   - Softmax: 转换为概率分布")
print("   - 决策: 选择概率最高的类别")
print()

print("5. 学习到的特征模式:")
print("   - 空间一致性: 所有箭头是否指向同一方向")
print("   - 对称性: Congruent刺激通常更对称")
print("   - 规律性: Congruent刺激的排列更有规律")
print("   - 冲突检测: Incongruent刺激中的方向冲突")

## 14. 特征可视化

In [ ]:
# 提取中间层特征进行可视化
def get_features(model, image_tensor, layer_name):
    """
    提取指定层的特征
    """
    features = []
    
    def hook_fn(module, input, output):
        features.append(output.detach())
    
    # 注册hook
    if layer_name == 'features':
        handle = model.features[-3].register_forward_hook(hook_fn)  # 最后一个卷积层
    
    # 前向传播
    with torch.no_grad():
        _ = model(image_tensor.unsqueeze(0))
    
    # 移除hook
    handle.remove()
    
    return features[0]

# 选择Congruent和Incongruent的样本
congruent_idx = np.where(np.array(sample_true_labels) == 0)[0][0]
incongruent_idx = np.where(np.array(sample_true_labels) == 1)[0][0]

# 获取特征
congruent_img_tensor = torch.from_numpy(sample_images[congruent_idx]).to(device)
incongruent_img_tensor = torch.from_numpy(sample_images[incongruent_idx]).to(device)

congruent_features = get_features(model, congruent_img_tensor, 'features')
incongruent_features = get_features(model, incongruent_img_tensor, 'features')

# 可视化特征图
fig, axes = plt.subplots(2, 8, figsize=(20, 6))

# Congruent样本特征
axes[0, 0].imshow(np.transpose(sample_images[congruent_idx], (1, 2, 0)))
axes[0, 0].set_title('Congruent输入', fontsize=12)
axes[0, 0].axis('off')

for i in range(1, 8):
    feat_map = congruent_features[0, i].cpu().numpy()
    axes[0, i].imshow(feat_map, cmap='viridis')
    axes[0, i].set_title(f'特征图{i}', fontsize=10)
    axes[0, i].axis('off')

# Incongruent样本特征
axes[1, 0].imshow(np.transpose(sample_images[incongruent_idx], (1, 2, 0)))
axes[1, 0].set_title('Incongruent输入', fontsize=12)
axes[1, 0].axis('off')

for i in range(1, 8):
    feat_map = incongruent_features[0, i].cpu().numpy()
    axes[1, i].imshow(feat_map, cmap='viridis')
    axes[1, i].set_title(f'特征图{i}', fontsize=10)
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

print("特征图说明：")
print("- 每个特征图检测不同的视觉模式")
print("- 亮色区域表示强激活")
print("- Congruent vs Incongruent的特征激活模式不同")
print("- 模型学习到区分一致性的关键特征")

## 15. 总结

In [ ]:
print("="*60)
print("Flanker任务一致性分类 - 总结")
print("="*60)
print()

print("✓ 完成的任务:")
print("  1. 生成了Flanker任务的视觉刺激图像")
print("  2. 使用预训练VGG16模型进行特征提取")
print("  3. 训练了一致性分类器")
print("  4. 实现了高准确率的一致性判断")
print()

print("✓ 模型判断一致性的方法:")
print("  - 视觉特征分析: 检测箭头方向的一致性")
print("  - 空间模式识别: 识别规律vs冲突的排列")
print("  - 深度学习: 自动学习区分性特征")
print("  - 端到端训练: 从原始图像到分类决策")
print()

print("✓ 关键发现:")
print(f"  - 测试准确率: {test_acc:.2f}%")
print("  - VGG特征提取器有效捕捉视觉模式")
print("  - 模型学会了区分Congruent和Incongruent刺激")
print("  - 为注意力研究提供了计算模型")
print()

print("✓ 应用价值:")
print("  - 认知科学研究: 量化注意力机制")
print("  - 临床应用: 检测注意力缺陷")
print("  - AI系统: 理解视觉冲突处理")
print("  - 脑机接口: 解码注意力状态")